In [1]:
from actor_critic import ActorCriticLSTM
from itertools import count
from collections import namedtuple
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Normal
import numpy as np
from tinyphysics import TinyPhysicsModel, TinyPhysicsSimulator, State, CONTEXT_LENGTH, STEER_RANGE, CONTROL_START_IDX, DEL_T, LAT_ACCEL_COST_MULTIPLIER, MAX_ACC_DELTA
from controllers import CONTROLLERS  # Ensure controllers.py is available and imported

In [2]:
SavedAction = namedtuple('SavedAction', ['log_prob', 'value'])

In [3]:
def select_action(target_lataccel, current_lataccel, state):
    state_tensor = torch.tensor([[target_lataccel, current_lataccel, state[0], state[1], state[2]]], dtype=torch.float32)
    state_tensor = state_tensor.unsqueeze(0)  # Add batch dimension

    mean_log_std, state_value = model(state_tensor)
    # Split mean and log_std
    mean, log_std = mean_log_std.chunk(2, dim=-1)
    # Exponentiate log to bring back to normal
    std = log_std.exp()
    dist = Normal(mean, std)
    action = dist.sample()
    model.saved_actions.append(SavedAction(dist.log_prob(action), state_value))
    return action.item()


In [4]:
def compute_costs(targets, predictions):
    lat_accel_costs = (targets - predictions) ** 2 * 100
    jerk_costs = (np.diff(predictions) / DEL_T) ** 2 * 100
    jerk_costs = np.insert(jerk_costs, 0, 0)  # Insert 0 jerk cost for the first timestep
    total_costs = (lat_accel_costs * LAT_ACCEL_COST_MULTIPLIER) + jerk_costs
    return lat_accel_costs, jerk_costs, total_costs


In [5]:
def finish_episode(gamma=0.99, eps=np.finfo(np.float32).eps.item()):
    R = 0
    policy_losses = []
    value_losses = []
    returns = []

    # Calculate costs
    targets = np.array(model.saved_targets)
    predictions = np.array(model.saved_action_results)
    _, _, total_costs = compute_costs(targets, predictions)

    # Convert costs to rewards
    rewards = -total_costs

    # Calculate cumulative returns
    for r in rewards[::-1]:
        R = r + gamma * R
        returns.insert(0, R)
    returns = torch.tensor(returns).to(next(model.parameters()).device)
    returns = (returns - returns.mean()) / (returns.std() + eps)

    # Calculate policy and value losses
    for (log_prob, value), R in zip(model.saved_actions, returns):
        advantage = R - value.item()
        policy_losses.append(-log_prob * advantage)
        value_losses.append(F.smooth_l1_loss(value, torch.tensor([[R]]).to(value.device)))

    # Perform backpropagation
    optimizer.zero_grad()
    loss = torch.stack(policy_losses).sum() + torch.stack(value_losses).sum()
    loss.backward()

    # Gradient clipping to prevent exploding gradients
    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    optimizer.step()
    schedular.step()

    # Clear saved actions, targets, and results for the next episode
    del model.saved_actions[:]
    del model.saved_targets[:]
    del model.saved_action_results[:]

In [6]:
def custom_update(target_lataccel, current_lataccel, state):
    if len(model.saved_targets) > 0:
        model.saved_action_results.append(current_lataccel)

    model.saved_targets.append(target_lataccel)

    # State order: roll_lataccel, v_ego, a_ego
    action = select_action(target_lataccel, current_lataccel, state)

    return action


In [7]:
def sim_step(sim):
    state, target = sim.get_state_target(sim.step_idx)
    sim.state_history.append(state)
    sim.target_lataccel_history.append(target)
    sim_control_step(sim, sim.step_idx)
    sim.sim_step(sim.step_idx)
    sim.step_idx += 1

def sim_control_step(sim, step_idx: int) -> None:
    if step_idx >= CONTROL_START_IDX:
      # Replace with custom call
      action = custom_update(sim.target_lataccel_history[step_idx], sim.current_lataccel, sim.state_history[step_idx])
    else:
      action = sim.data['steer_command'].values[step_idx]
    action = np.clip(action, STEER_RANGE[0], STEER_RANGE[1])
    sim.action_history.append(action)

In [8]:
def compute_summary(model) -> dict:
    target = np.array(model.saved_targets)
    pred = np.array(model.saved_action_results)

    lat_accel_cost = np.mean((target - pred)**2) * 100
    jerk_cost = np.mean((np.diff(pred) / DEL_T)**2) * 100
    total_cost = (lat_accel_cost * LAT_ACCEL_COST_MULTIPLIER) + jerk_cost
    return {'lataccel_cost': lat_accel_cost, 'jerk_cost': jerk_cost, 'total_cost': total_cost}

In [9]:
model = ActorCriticLSTM()
model.init_hidden(batch_size=1)
optimizer = optim.Adam(model.parameters(), lr=3e-2)
schedular = optim.lr_scheduler.StepLR(optimizer, 20, 0.5)
eps = np.finfo(np.float32).eps.item()

In [10]:
def train():
    model_path = "./models/tinyphysics.onnx"
    data_path = "./data/00000.csv"
    controller_name = "simple"
    debug = False

    tinyphysics_model = TinyPhysicsModel(model_path, debug)
    controller = CONTROLLERS[controller_name]()
    sim = TinyPhysicsSimulator(tinyphysics_model, data_path, controller, debug)

    for i in range(200):
        sim.reset()
        model.init_hidden(1)
        for _ in range(CONTEXT_LENGTH, len(sim.data)):
            sim_step(sim)
        
        # Need to remove last episode data as the resultant lataccel isn't accessible
        model.saved_actions.pop()
        model.saved_targets.pop()

        print(compute_summary(model))

        finish_episode()

In [11]:
train()

2024-05-17 18:24:12.334026445 [E:onnxruntime:Default, provider_bridge_ort.cc:1480 TryGetProviderInfo_CUDA] /onnxruntime_src/onnxruntime/core/session/provider_bridge_ort.cc:1193 onnxruntime::Provider& onnxruntime::ProviderLibrary::Get() [ONNXRuntimeError] : 1 : FAIL : Failed to load library libonnxruntime_providers_cuda.so with error: libcublasLt.so.11: cannot open shared object file: No such file or directory

2024-05-17 18:24:12.334041153 [W:onnxruntime:Default, onnxruntime_pybind_state.cc:747 CreateExecutionProviderInstance] Failed to create CUDAExecutionProvider. Please reference https://onnxruntime.ai/docs/execution-providers/CUDA-ExecutionProvider.html#requirements to ensure all dependencies are met.


{'lataccel_cost': 94.12936017818211, 'jerk_cost': 2045.0899821176308, 'total_cost': 2515.7367830085414}
{'lataccel_cost': 319.615156259094, 'jerk_cost': 2280.3248638094024, 'total_cost': 3878.4006451048726}
{'lataccel_cost': 113.63607346536513, 'jerk_cost': 2258.6969135964287, 'total_cost': 2826.877280923254}
{'lataccel_cost': 149.30878995337756, 'jerk_cost': 2145.155029426958, 'total_cost': 2891.698979193846}
{'lataccel_cost': 115.23087858101464, 'jerk_cost': 2075.3796454181, 'total_cost': 2651.5340383231733}
{'lataccel_cost': 88.08429763774356, 'jerk_cost': 1925.6947553546113, 'total_cost': 2366.116243543329}
{'lataccel_cost': 80.6265849213304, 'jerk_cost': 1998.8448062355649, 'total_cost': 2401.9777308422167}
{'lataccel_cost': 81.05566573179746, 'jerk_cost': 1827.578649253771, 'total_cost': 2232.8569779127583}
{'lataccel_cost': 85.11551993518323, 'jerk_cost': 1767.477615360206, 'total_cost': 2193.0552150361223}
{'lataccel_cost': 91.01022576439652, 'jerk_cost': 1861.2909770677727, 't

In [12]:
torch.save(model, './models/actor_critic.pth')